# Análise de Dados

## 0. Colab

In [1]:
import os
import shutil
import pandas as pd
from google.colab import userdata, drive

drive.mount('/content/drive/')

os.chdir("/content/drive/MyDrive/MD")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
df_densidade = pd.read_csv(os.path.join("yolo_features_densidade.csv"))
df_tamanho = pd.read_csv(os.path.join("yolo_features_tamanho.csv"))

## 1. Atividade

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import math
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.ensemble import IsolationForest
from pandas.plotting import parallel_coordinates

RANDOM_STATE = 42

def info_basica(df, class_names):
    # 1. Informações Básicas
    print(f"Total de amostras: {len(df)}")
    print(f"Classes encontradas: {list(class_names)}")
    print()

    for class_name in class_names:
        print(f"Classe: {class_name}")
        len_class= len(df[df['class_name'] == class_name])
        print(f"Total de amostras: {len_class}")
        print(f"Porcentagem das amostras: {len_class/len(df) * 100:.2f}%.")
        print()

def visu_imagens(df, nome_dataset, class_names):
    # 2. Visualização de Imagens (4 exemplos por classe)
    print(f"--- 1. Amostra de Imagens Brutas ({nome_dataset}) ---")
    n_examples = 4
    fig, axes = plt.subplots(len(class_names), n_examples, figsize=(12, 3 * len(class_names)))

    # Garantir que axes seja 2D mesmo com 1 classe
    if len(class_names) == 1: axes = np.expand_dims(axes, axis=0)

    for class_id, class_name in enumerate(class_names):
        class_items = df[df["class_name"] == class_name].head(n_examples)
        for j, (_, row) in enumerate(class_items.iterrows()):
            img_path = row["file_path"]
            ax = axes[class_id, j]
            try:
                img = Image.open(img_path)
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, 'Erro ao carregar', ha='center', va='center')
            ax.axis("off")
            ax.set_title(class_name)
    plt.tight_layout()
    plt.show()
    print()

def escalonamento(df):
    # 3. Preparação e Escalonamento (Z-score)
    feature_cols = [col for col in df.columns if col.startswith('f_')]
    X_img = df[feature_cols].values
    y_labels = df["class_name"].values

    X_img_scaled = StandardScaler().fit_transform(X_img)
    print(f"Formato dos embeddings: {X_img_scaled.shape[0]} amostras x {X_img_scaled.shape[1]} features.")
    print()

    return feature_cols, X_img_scaled, y_labels

def eda(nome_dataset, class_names, feature_cols, X_img_scaled, y_labels):
    # 4. EDA - Parallel Coordinates (30 features aleatórias)
    print(f"--- 2. Parallel Coordinates (30 features aleatórias) ---")
    rng = np.random.default_rng(RANDOM_STATE)
    selected_feature_idx = np.sort(rng.choice(X_img_scaled.shape[1], size=30, replace=False))
    selected_feature_names = [feature_cols[i] for i in selected_feature_idx]

    df_parallel_20 = pd.DataFrame(X_img_scaled[:, selected_feature_idx], columns=selected_feature_names)
    df_parallel_20["class_name"] = y_labels

    n_samples_per_class = 10
    parts = [group.sample(min(len(group), n_samples_per_class), random_state=RANDOM_STATE)
             for _, group in df_parallel_20.groupby("class_name")]
    sample_parallel_20 = pd.concat(parts).reset_index(drop=True)

    colors = [plt.cm.tab10(i) for i in range(len(class_names))]

    plt.figure(figsize=(16, 6))
    parallel_coordinates(sample_parallel_20, class_column="class_name", color=colors, alpha=0.6, linewidth=1.5)
    plt.title(f"YOLO26 — 30 features ({nome_dataset})", fontsize=15)
    plt.xlabel("Features")
    plt.ylabel("Z-score")
    plt.xticks(rotation=70)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print()

def pca(nome_dataset, X_img_scaled):
    # 5. PCA - Variância Explicada (8 Componentes)
    print(f"--- 3. PCA: Variância Explicada (8 PCs) ---")
    pca_img = PCA(n_components=8, random_state=RANDOM_STATE)
    pca_img.fit(X_img_scaled)
    var_acumulada = np.cumsum(pca_img.explained_variance_ratio_)

    plt.figure(figsize=(7,4))
    plt.plot(range(1, 9), var_acumulada, marker="o")
    plt.xlabel("Número de componentes")
    plt.ylabel("Variância explicada acumulada")
    plt.title(f"PCA — Variância explicada ({nome_dataset})")
    plt.grid(alpha=0.3)
    plt.show()
    print()

def pca_3d(nome_dataset, X_img_scaled, y_labels):
    # 6. PCA em 3 Dimensões (Plotly Interativo)
    print(f"--- 4. PCA em 3 Dimensões ---")
    pca_img_3d = PCA(n_components=3, random_state=RANDOM_STATE)
    X_img_pca3 = pca_img_3d.fit_transform(X_img_scaled)

    df_pca3 = pd.DataFrame(X_img_pca3, columns=["PC1", "PC2", "PC3"])
    df_pca3["class_name"] = y_labels

    fig = px.scatter_3d(
        df_pca3, x="PC1", y="PC2", z="PC3", color="class_name", opacity=0.8,
        title=f"YOLO26 — PCA 3D ({nome_dataset})"
    )
    fig.update_traces(marker=dict(size=5))
    fig.update_layout(width=900, height=600, margin=dict(l=0, r=0, b=0, t=40))
    fig.show()
    print()

def tsne(nome_dataset, X_img_scaled, y_labels):
    # 7. TSNE
    print(f"--- 5. TSNE ({nome_dataset}) ---")

    X_img_tsne = TSNE(n_components=2, random_state=RANDOM_STATE).fit_transform(X_img_scaled)

    print(f"Shape do embedding TSNE: {X_img_tsne.shape}")
    df_tsne = pd.DataFrame(X_img_tsne, columns=["TSNE1", "TSNE2"])
    df_tsne["class_name"] = y_labels

    fig = px.scatter(
        df_tsne, x="TSNE1", y="TSNE2", color="class_name", opacity=0.8,
        title=f"YOLO26 — TSNE ({nome_dataset})"
    )
    fig.show()
    print()

def umap_fuction(nome_dataset, X_img_scaled, y_labels):
    # 8. UMAP
    print(f"--- 6. UMAP ({nome_dataset}) ---")

    reducer = umap.UMAP()
    X_img_umap = reducer.fit_transform(X_img_scaled)

    print(f"Shape do embedding UMAP: {X_img_umap.shape}")

    df_umap = pd.DataFrame(X_img_umap, columns=["UMAP1", "UMAP2"])
    df_umap["class_name"] = y_labels

    fig = px.scatter(
        df_umap, x="UMAP1", y="UMAP2", color="class_name", opacity=0.8,
        title=f"YOLO26 — UMAP ({nome_dataset})"
    )
    fig.show()
    print()

def outliers(df, X_img_scaled):
    print(f"--- 7. Detecção de Anomalias/Outliers ---")
    iso_img = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
    img_outlier_pred = iso_img.fit_predict(X_img_scaled)
    outlier_positions = np.where(img_outlier_pred == -1)[0]

    n_outliers = len(outlier_positions)
    print(f"Candidatos a outlier encontrados: {n_outliers}")

    if n_outliers > 0:
        n_cols = 4
        n_rows = math.ceil(n_outliers / n_cols)

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3), squeeze=False)
        axes_flat = axes.flat

        for ax, pos in zip(axes_flat, outlier_positions):
            row = df.iloc[pos]
            try:
                img = Image.open(row["file_path"])
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, 'Erro', ha='center', va='center')
            ax.set_title(str(row["class_name"]))
            ax.axis("off")

        for ax in axes_flat[n_outliers:]:
            ax.axis("off")

        plt.tight_layout()
        plt.show()
        print()

def analisar_dataset_yolo(df, nome_dataset, bool_info, bool_visu, bool_eda, bool_pca, bool_pca_3d, bool_tsne, bool_umap, bool_outliers):
    print(f"{'='*50}")
    print(f"INICIANDO ANÁLISE: DATASET DE {nome_dataset.upper()}")
    print(f"{'='*50}")

    class_names = df["class_name"].unique()
    feature_cols, X_img_scaled, y_labels = escalonamento(df)

    if bool_info:
      info_basica(df, class_names)

    if bool_visu:
      visu_imagens(df, nome_dataset, class_names)

    if bool_eda:
      eda(nome_dataset, class_names, feature_cols, X_img_scaled, y_labels)

    if bool_pca:
      pca(nome_dataset, X_img_scaled)

    if bool_pca_3d:
      pca_3d(nome_dataset, X_img_scaled, y_labels)

    if bool_tsne:
      tsne(nome_dataset, X_img_scaled, y_labels)

    if bool_umap:
      umap_fuction(nome_dataset, X_img_scaled, y_labels)

    if bool_outliers:
      outliers(df, X_img_scaled)

    print("Análise Concluída!")

ValueError: Name tf.RaggedTensorSpec has already been registered for class tensorflow.python.ops.ragged.ragged_tensor.RaggedTensorSpec.

In [ ]:
analisar_dataset_yolo(df_densidade, "Densidade", True, True, True, True, True, True, True, True)

In [ ]:
analisar_dataset_yolo(df_tamanho, "Tamanho", True, True, True, True, True, True, True, True)

## 2. Prova 1

### 2.1 Base

In [3]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from scipy.spatial.distance import pdist, cdist
from scipy.optimize import linear_sum_assignment
import umap

RANDOM_STATE = 42
CORES = [plt.cm.tab10(i) for i in range(10)]

### 2.2 Auxiliar

In [4]:
# FUNÇÕES AUXILIARES
def mostrar_imagens(df, idxs, titulos, n_cols=4, suptitle=None):
    """Grade de imagens a partir de posições (iloc) do dataframe."""
    n = len(idxs)
    if n == 0:
        print("Nenhuma imagem para mostrar.")
        return
    n_rows = math.ceil(n / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i, t in zip(axes.flat, idxs, titulos):
        try:
            ax.imshow(Image.open(df.iloc[i]["file_path"]))
        except Exception:
            ax.text(0.5, 0.5, "Erro ao carregar", ha="center", va="center")
        ax.set_title(t, fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    plt.show()


def scatter_cat(ax, coords, cats, title):
    """Scatter 2D colorido por categoria (rótulos ou clusters)."""
    cats = np.asarray(cats)
    for i, c in enumerate(sorted(pd.unique(cats), key=str)):
        m = cats == c
        ax.scatter(coords[m, 0], coords[m, 1], s=18, alpha=0.75,
                   color=CORES[i % 10], label=str(c))
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def hopkins(X, frac=0.1, seed=RANDOM_STATE):
    """Estatística de Hopkins: ~0.5 = dados aleatórios/uniformes; >0.75 = tendência a clusters."""
    rng = np.random.default_rng(seed)
    n, d = X.shape
    m = max(5, int(frac * n))
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    idx = rng.choice(n, m, replace=False)
    w = nn.kneighbors(X[idx], n_neighbors=2)[0][:, 1]
    U = rng.uniform(X.min(0), X.max(0), (m, d))
    u = nn.kneighbors(U, n_neighbors=1)[0][:, 0]
    return u.sum() / (u.sum() + w.sum())


def alinhar_clusters(a, b):
    """Reordena os ids de b para casar o máximo possível com a (húngaro). Só p/ mesmo k."""
    ct = pd.crosstab(a, b)
    r, c = linear_sum_assignment(-ct.values)
    mapa = {ct.columns[j]: ct.index[i] for i, j in zip(r, c)}
    return np.array([mapa.get(x, -1) for x in b])


def combinar_rotulos(df_dens, df_tam):
    """Junta os dois datasets (mesmas imagens) em um com 6 categorias 'Denso|Pequena', etc.
    O casamento é feito pelo nome do arquivo."""
    a = df_dens.copy(); b = df_tam.copy()
    a["_nome"] = a["file_path"].apply(lambda p: str(p).replace("\\", "/").split("/")[-1])
    b["_nome"] = b["file_path"].apply(lambda p: str(p).replace("\\", "/").split("/")[-1])
    m = a.merge(b[["_nome", "class_name"]], on="_nome", suffixes=("_dens", "_tam"))
    m["class_name"] = m["class_name_dens"] + " | " + m["class_name_tam"]
    print(f"Imagens casadas: {len(m)} de {len(a)} (dens) / {len(b)} (tam)")
    return m.drop(columns=["_nome", "class_name_dens", "class_name_tam"])

### 2.3 Preparação

In [5]:
# PREPARAÇÃO
def info_basica(df, class_names):
    print(f"Total de amostras: {len(df)}")
    print(f"Classes encontradas: {list(class_names)}\n")
    for c in class_names:
        n = (df["class_name"] == c).sum()
        print(f"Classe: {c} | amostras: {n} ({n/len(df)*100:.2f}%)")
    print()


def visu_imagens(df, nome_dataset, class_names, n_examples=4):
    print(f"--- Amostra de imagens brutas ({nome_dataset}) ---")
    fig, axes = plt.subplots(len(class_names), n_examples,
                             figsize=(3 * n_examples, 3 * len(class_names)), squeeze=False)
    for i, c in enumerate(class_names):
        itens = df[df["class_name"] == c].head(n_examples)
        for j in range(n_examples):
            ax = axes[i, j]; ax.axis("off")
            if j < len(itens):
                try:
                    ax.imshow(Image.open(itens.iloc[j]["file_path"]))
                except Exception:
                    ax.text(0.5, 0.5, "Erro", ha="center", va="center")
                ax.set_title(c, fontsize=9)
    plt.tight_layout(); plt.show()


def preparar(df, n_pca=50, usar_l2=True):
    """
    Retorna:
      X_pca  : representação usada em K-Means / hierárquico (z-score -> L2 -> PCA)
      y      : rótulos (somente para interpretação)
    """
    feature_cols = [c for c in df.columns if c.startswith("f_")]
    X = df[feature_cols].values.astype(float)
    print(f"Embeddings: {X.shape[0]} amostras x {X.shape[1]} features")
    print(f"Valores faltantes: {np.isnan(X).sum()}")

    const = X.std(0) < 1e-8
    if const.any():
        print(f"Removendo {const.sum()} features constantes")
        X = X[:, ~const]

    X = StandardScaler().fit_transform(X)
    if usar_l2:
        X = normalize(X)

    n_comp = min(n_pca, X.shape[0] - 1, X.shape[1])
    pca_ = PCA(n_components=n_comp, random_state=RANDOM_STATE).fit(X)
    X_pca = pca_.transform(X)
    var = np.cumsum(pca_.explained_variance_ratio_)
    print(f"PCA {n_comp} componentes explica {var[-1]*100:.1f}% da variância "
          f"(PC1={pca_.explained_variance_ratio_[0]*100:.1f}%, "
          f"10 PCs={var[min(9, n_comp-1)]*100:.1f}%)\n")
    return X_pca, df["class_name"].values


def tendencia_a_clusters(X, y):
    """Evidência prévia: os dados têm tendência a formar grupos? E os rótulos separam o espaço?"""
    h = hopkins(X[:, :10])
    print(f"Hopkins (10 PCs): {h:.3f}  (0.5 ≈ sem estrutura; >0.75 ≈ clusterizável)")
    if len(set(y)) > 1:
        s = silhouette_score(X, y)
        print(f"Silhouette usando os PRÓPRIOS RÓTULOS como grupos: {s:.3f} "
              f"(≈0 ou negativo => rótulos não formam regiões separadas no embedding)\n")

### 2.4 Redução, K-Means e Hirarquia

In [6]:
# 2. REDUÇÃO DE DIMENSIONALIDADE
def reducao(nome_dataset, X, y, perplexities=(10, 30, 50), neighbors=(5, 15, 50)):
    """PCA 2D + t-SNE e UMAP com vários hiperparâmetros (testa estabilidade da estrutura).
    Retorna dict com coordenadas 'padrão' para reusar nos gráficos de clusters."""
    coords = {}

    p2 = PCA(n_components=2, random_state=RANDOM_STATE).fit(X)
    coords["pca"] = p2.transform(X)
    fig, ax = plt.subplots(figsize=(6, 5))
    scatter_cat(ax, coords["pca"], y, f"PCA 2D ({nome_dataset}) — "
                f"var. {p2.explained_variance_ratio_.sum()*100:.1f}%")
    ax.legend(fontsize=8); plt.show()

    # t-SNE
    perps = [p for p in perplexities if p < len(X) / 3] or [max(2, len(X) // 4)]
    fig, axes = plt.subplots(1, len(perps), figsize=(5.5 * len(perps), 5), squeeze=False)
    for ax, p in zip(axes[0], perps):
        emb = TSNE(n_components=2, perplexity=p, init="pca", random_state=RANDOM_STATE).fit_transform(X)
        scatter_cat(ax, emb, y, f"t-SNE perplexity={p}")
        coords[f"tsne_{p}"] = emb
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f"t-SNE ({nome_dataset})"); plt.tight_layout(); plt.show()
    coords["tsne"] = coords[f"tsne_{perps[len(perps)//2]}"]

    # UMAP
    nbs = [n for n in neighbors if n < len(X)] or [2]
    fig, axes = plt.subplots(1, len(nbs), figsize=(5.5 * len(nbs), 5), squeeze=False)
    for ax, n in zip(axes[0], nbs):
        emb = umap.UMAP(n_neighbors=n, n_jobs=-1, random_state=RANDOM_STATE).fit_transform(X)
        scatter_cat(ax, emb, y, f"UMAP n_neighbors={n}")
        coords[f"umap_{n}"] = emb
    axes[0][0].legend(fontsize=8)
    fig.suptitle(f"UMAP ({nome_dataset})"); plt.tight_layout(); plt.show()
    coords["umap"] = coords[f"umap_{nbs[len(nbs)//2]}"]

    print("Atenção: t-SNE/UMAP distorcem distâncias globais; avalie apenas se as "
          "ilhas persistem entre diferentes hiperparâmetros.\n")
    return coords


# 2. K-MEANS
def kmeans_analise(nome_dataset, X, k_min=2, k_max=10):
    """Cotovelo + silhouette + Davies-Bouldin + Calinski-Harabasz."""
    linhas = []
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
        linhas.append(dict(k=k, inercia=km.inertia_,
                           silhouette=silhouette_score(X, km.labels_),
                           davies_bouldin=davies_bouldin_score(X, km.labels_),
                           calinski_harabasz=calinski_harabasz_score(X, km.labels_)))
    res = pd.DataFrame(linhas).set_index("k")

    fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
    for ax, (col, tit, dica) in zip(axes, [
            ("inercia", "Cotovelo (inércia)", "procure o 'joelho'"),
            ("silhouette", "Silhouette", "maior = melhor"),
            ("davies_bouldin", "Davies-Bouldin", "menor = melhor"),
            ("calinski_harabasz", "Calinski-Harabasz", "maior = melhor")]):
        ax.plot(res.index, res[col], marker="o"); ax.set_title(f"{tit}\n({dica})", fontsize=10)
        ax.set_xlabel("k"); ax.grid(alpha=0.3)
    fig.suptitle(f"K-Means — escolha de k ({nome_dataset})"); plt.tight_layout(); plt.show()

    print(res.round(3))
    print(f"\nMelhor k por silhouette: {res['silhouette'].idxmax()} "
          f"(silhouette={res['silhouette'].max():.3f})")
    print("Referência: silhouette < 0.25 indica estrutura fraca/sobreposta.\n")
    return res


def kmeans_resultado(nome_dataset, X, y, k, coords):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    lab = km.labels_
    print(f"--- K-Means k={k} ({nome_dataset}) ---")
    print(f"ARI vs rótulos: {adjusted_rand_score(y, lab):.3f} | "
          f"NMI vs rótulos: {normalized_mutual_info_score(y, lab):.3f}")
    print("\nTabela de contingência (cluster x rótulo):")
    print(pd.crosstab(lab, y), "\n")

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ax, key, nm in zip(axes, ["pca", "tsne", "umap"], ["PCA", "t-SNE", "UMAP"]):
        scatter_cat(ax, coords[key], [f"C{c}" for c in lab], f"{nm} — clusters K-Means (k={k})")
    axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()
    return km, lab


# 3. HIERÁRQUICO
def hierarquico(nome_dataset, X, y, k, lab_km=None, k_max_gap=10):
    """Ward (euclidiana) + average (cosseno): dendrogramas, saltos de altura, cophenetic."""
    Zw = linkage(X, method="ward", metric="euclidean")
    Za = linkage(X, method="average", metric="cosine")
    cw = cophenet(Zw, pdist(X))[0]
    ca = cophenet(Za, pdist(X, metric="cosine"))[0]
    print(f"Correlação cofenética — Ward/euclid: {cw:.3f} | Average/cosseno: {ca:.3f} "
          f"(mais próximo de 1 = dendrograma preserva melhor as distâncias)")

    # Dendrogramas
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    thr = Zw[-(k - 1), 2] - 1e-9 if k > 1 else None
    dendrogram(Zw, truncate_mode="lastp", p=30, ax=axes[0], color_threshold=thr,
               show_leaf_counts=True)
    axes[0].axhline(thr, ls="--", c="k", lw=1)
    axes[0].set_title(f"Ward / euclidiana — corte em k={k}")
    dendrogram(Za, truncate_mode="lastp", p=30, ax=axes[1], show_leaf_counts=True)
    axes[1].set_title("Average / cosseno")
    for ax in axes:
        ax.set_ylabel("Altura de fusão")
    fig.suptitle(f"Dendrogramas ({nome_dataset})"); plt.tight_layout(); plt.show()

    # Saltos de altura (Ward)
    ks = np.arange(2, k_max_gap + 1)
    alt = np.array([Zw[-(kk - 1), 2] for kk in ks])      # altura da fusão k -> k-1
    alt_prev = np.array([Zw[-kk, 2] for kk in ks])       # altura da fusão (k+1) -> k
    gap = alt - alt_prev                                  # salto ao ir de k para k-1
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.bar(ks, gap); ax.set_xlabel("k (número de clusters ANTES da fusão)")
    ax.set_ylabel("Salto de altura"); ax.set_title("Saltos Ward: pico em k sugere cortar em k")
    ax.grid(alpha=0.3); plt.show()
    print(f"k sugerido pelo maior salto (Ward, k>=2): {ks[np.argmax(gap)]}\n")

    # Corte
    lab_h = fcluster(Zw, t=k, criterion="maxclust")
    print(f"--- Hierárquico Ward k={k} ---")
    print(f"ARI vs rótulos: {adjusted_rand_score(y, lab_h):.3f} | "
          f"NMI vs rótulos: {normalized_mutual_info_score(y, lab_h):.3f}")
    if lab_km is not None:
        print(f"ARI K-Means x Hierárquico: {adjusted_rand_score(lab_km, lab_h):.3f}")
    print("\nTabela de contingência (cluster x rótulo):")
    print(pd.crosstab(lab_h, y), "\n")

    lab_avg = fcluster(Za, t=k, criterion="maxclust")
    print(f"ARI Ward x Average (k={k}): {adjusted_rand_score(lab_h, lab_avg):.3f}\n")
    return Zw, lab_h


def distancia_entre_centroides(nome, X, lab, titulo):
    ids = np.unique(lab)
    C = np.vstack([X[lab == c].mean(0) for c in ids])
    D = cdist(C, C)
    fig, ax = plt.subplots(figsize=(4.5 + 0.3 * len(ids), 4 + 0.3 * len(ids)))
    im = ax.imshow(D, cmap="viridis")
    ax.set_xticks(range(len(ids))); ax.set_xticklabels([f"C{i}" for i in ids])
    ax.set_yticks(range(len(ids))); ax.set_yticklabels([f"C{i}" for i in ids])
    for i in range(len(ids)):
        for j in range(len(ids)):
            ax.text(j, i, f"{D[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
    ax.set_title(f"Distância entre centróides — {titulo} ({nome})")
    plt.colorbar(im); plt.tight_layout(); plt.show()

### 2.5 Análise

In [7]:
# 5. INTERPRETAÇÃO
def interpretar_clusters(df, nome_dataset, X, lab, y, titulo, n_img=4):
    """Protótipos (mais próximos do centróide) + distribuição dos rótulos por cluster."""
    ids = np.unique(lab)
    dist_pct = pd.crosstab(lab, y, normalize="index") * 100
    print(f"--- Distribuição de rótulos por cluster — {titulo} ({nome_dataset}) [%] ---")
    print(dist_pct.round(1), "\n")
    print("Tamanho dos clusters:", dict(zip(*np.unique(lab, return_counts=True))), "\n")

    ax = dist_pct.plot(kind="bar", stacked=True, figsize=(7, 4), colormap="tab10")
    ax.set_ylabel("% das imagens do cluster"); ax.set_xlabel("cluster")
    ax.set_title(f"Composição dos clusters — {titulo} ({nome_dataset})")
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1)); plt.tight_layout(); plt.show()

    idxs, tits = [], []
    for c in ids:
        pos = np.where(lab == c)[0]
        cen = X[pos].mean(0)
        ordem = pos[np.argsort(np.linalg.norm(X[pos] - cen, axis=1))][:n_img]
        for p in ordem:
            idxs.append(p); tits.append(f"C{c} | {y[p]}")
        idxs += [None] * (n_img - len(ordem)); tits += [""] * (n_img - len(ordem))
    # grade: uma linha por cluster
    fig, axes = plt.subplots(len(ids), n_img, figsize=(3 * n_img, 3 * len(ids)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i, t in zip(axes.flat, idxs, tits):
        if i is None:
            continue
        try:
            ax.imshow(Image.open(df.iloc[i]["file_path"]))
        except Exception:
            ax.text(0.5, 0.5, "Erro", ha="center", va="center")
        ax.set_title(t, fontsize=9)
    fig.suptitle(f"Protótipos de cada cluster — {titulo} ({nome_dataset})")
    plt.tight_layout(); plt.show()


# 6. EXEMPLOS QUE NÃO SE ENCAIXAM
def casos_problematicos(df, nome_dataset, X, y, lab_km, lab_h, n=8):
    # (a) silhouette baixo/negativo
    sil = silhouette_samples(X, lab_km)
    print(f"Imagens com silhouette negativo no K-Means: {(sil < 0).sum()} de {len(sil)} "
          f"({(sil < 0).mean()*100:.1f}%)")
    pior = np.argsort(sil)[:n]
    mostrar_imagens(df, pior, [f"C{lab_km[i]} | {y[i]}\nsil={sil[i]:.2f}" for i in pior],
                    suptitle=f"Menor silhouette ({nome_dataset})")

    # (b) discordância K-Means x hierárquico
    if len(np.unique(lab_km)) == len(np.unique(lab_h)):
        h_al = alinhar_clusters(lab_km, lab_h)
        disc = np.where(h_al != lab_km)[0]
        print(f"Imagens em que K-Means e hierárquico discordam: {len(disc)} "
              f"({len(disc)/len(lab_km)*100:.1f}%)")
        sel = disc[np.argsort(sil[disc])][:n]
        mostrar_imagens(df, sel, [f"KM=C{lab_km[i]} / H=C{h_al[i]} | {y[i]}" for i in sel],
                        suptitle=f"Discordâncias K-Means x Hierárquico ({nome_dataset})")

    # (c) outliers: longe do próprio centróide
    d = np.zeros(len(X))
    for c in np.unique(lab_km):
        m = lab_km == c
        d[m] = np.linalg.norm(X[m] - X[m].mean(0), axis=1)
    out = np.argsort(-d)[:n]
    mostrar_imagens(df, out, [f"C{lab_km[i]} | {y[i]}\nd={d[i]:.2f}" for i in out],
                    suptitle=f"Mais distantes do centróide ({nome_dataset})")


# EXECUÇÃO
def analisar_dataset_yolo(df, nome_dataset, k_kmeans=3, k_hier=3,
                          bool_info=True, bool_visu=True, bool_reducao=True,
                          bool_kmeans=True, bool_hier=True, bool_interp=True,
                          bool_casos=True, n_pca=50, usar_l2=True):
    print("=" * 60)
    print(f"INICIANDO ANÁLISE: {nome_dataset.upper()}")
    print("=" * 60)
    df = df.reset_index(drop=True)
    class_names = df["class_name"].unique()

    if bool_info:
        info_basica(df, class_names)
    if bool_visu:
        visu_imagens(df, nome_dataset, class_names)

    X, y = preparar(df, n_pca=n_pca, usar_l2=usar_l2)
    tendencia_a_clusters(X, y)

    coords = reducao(nome_dataset, X, y) if bool_reducao else None

    res = dict(X=X, y=y, coords=coords)
    if bool_kmeans:
        res["metricas_km"] = kmeans_analise(nome_dataset, X)
        if coords is not None:
            km, lab_km = kmeans_resultado(nome_dataset, X, y, k_kmeans, coords)
            res["lab_km"] = lab_km
    if bool_hier:
        Z, lab_h = hierarquico(nome_dataset, X, y, k_hier, res.get("lab_km"))
        res["lab_h"] = lab_h
        distancia_entre_centroides(nome_dataset, X, lab_h, f"hierárquico k={k_hier}")
    if bool_interp:
        if "lab_km" in res:
            interpretar_clusters(df, nome_dataset, X, res["lab_km"], y, f"K-Means k={k_kmeans}")
        if "lab_h" in res:
            interpretar_clusters(df, nome_dataset, X, res["lab_h"], y, f"Hierárquico k={k_hier}")
    if bool_casos and "lab_km" in res and "lab_h" in res:
        casos_problematicos(df, nome_dataset, X, y, res["lab_km"], res["lab_h"])

    print("Análise concluída!")
    return res

### 2.6 Resultado

In [ ]:
res_dens = analisar_dataset_yolo(df_densidade, "Densidade", k_kmeans=2, k_hier=2)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
res_tam  = analisar_dataset_yolo(df_tamanho, "Tamanho", k_kmeans=3, k_hier=3)

Output hidden; open in https://colab.research.google.com to view.

## 3. Teste

In [12]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

RANDOM_STATE = 42

def balancear(df, metodo="aleatorio", n_alvo=None, random_state=RANDOM_STATE, verbose=True):
    """
    Balanceia o dataset reduzindo as classes maiores até n_alvo.

    Parâmetros:
        df          : DataFrame com colunas 'class_name' e features 'f_*'
        metodo      : 'aleatorio' (subamostragem sem viés - RECOMENDADO),
                      'isolation' (mantém menos outliers via IsolationForest - ENVIÉSADO) ou
                      'centroide' (mantém mais próximos ao centroide - ENVIÉSADO)
        n_alvo      : nº de amostras por classe. Se None, usa o tamanho da menor classe.
        random_state: semente para reprodutibilidade
        verbose     : imprime a contagem antes/depois
    """
    y = df["class_name"].values
    contagem = df["class_name"].value_counts()

    if n_alvo is None:
        n_alvo = contagem.min()

    if verbose:
        print("Antes: ", contagem.to_dict())

    rng = np.random.default_rng(random_state)
    manter = np.zeros(len(df), dtype=bool)

    # Só calcula StandardScaler se realmente for usar distância/IsolationForest
    if metodo in ("isolation", "centroide"):
        feature_cols = [c for c in df.columns if c.startswith("f_")]
        X = StandardScaler().fit_transform(df[feature_cols].values)

    for classe in contagem.index:
        pos = np.where(y == classe)[0]

        # Classe pequena o bastante: mantém tudo
        if len(pos) <= n_alvo:
            manter[pos] = True
            continue

        if metodo == "aleatorio":
            # Sorteia n_alvo amostras sem reposição (mantém a distribuição original da classe)
            escolhidos = rng.choice(pos, size=n_alvo, replace=False)
            manter[escolhidos] = True

        elif metodo in ("isolation", "centroide"):
            Xc = X[pos]
            if metodo == "isolation":
                iso = IsolationForest(random_state=random_state).fit(Xc)
                score = -iso.score_samples(Xc)  # maior = mais outlier
            else:
                score = np.linalg.norm(Xc - Xc.mean(axis=0), axis=1)

            # Fica com as n_alvo amostras menos outliers (introduz viés de seleção!)
            mais_tipicos = pos[np.argsort(score)[:n_alvo]]
            manter[mais_tipicos] = True
        else:
            raise ValueError("metodo deve ser 'aleatorio', 'isolation' ou 'centroide'")

    df_bal = df[manter].reset_index(drop=True)

    if verbose:
        print(f"Depois ({metodo}):", df_bal["class_name"].value_counts().to_dict())
        print(f"Removidos: {len(df) - len(df_bal)} de {len(df)}\n")

    return df_bal

In [13]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score

class UnderSamplingClassifier(BaseEstimator, ClassifierMixin):
    """
    Wrapper que aplica o StandardScaler e o subamostrador APENAS nos dados de treino
    de cada fold da validação cruzada, evitando vazamento de dados (data leakage)
    e mantendo o conjunto de teste intacto com a distribuição real.
    """
    def __init__(self, base_estimator=None, metodo="aleatorio", random_state=42):
        self.base_estimator = base_estimator
        self.metodo = metodo
        self.random_state = random_state

    def fit(self, X, y):
        self.scaler_ = StandardScaler()
        X_scaled = self.scaler_.fit_transform(X)

        classes, counts = np.unique(y, return_counts=True)
        self.classes_ = classes
        n_alvo = counts.min()

        rng = np.random.default_rng(self.random_state)
        manter = np.zeros(len(y), dtype=bool)

        for c in classes:
            pos = np.where(y == c)[0]
            if len(pos) <= n_alvo:
                manter[pos] = True
                continue

            if self.metodo == "aleatorio":
                escolhidos = rng.choice(pos, size=n_alvo, replace=False)
                manter[escolhidos] = True
            elif self.metodo == "isolation":
                Xc = X_scaled[pos]
                iso = IsolationForest(random_state=self.random_state).fit(Xc)
                score = -iso.score_samples(Xc)
                manter[pos[np.argsort(score)[:n_alvo]]] = True
            elif self.metodo == "centroide":
                Xc = X_scaled[pos]
                score = np.linalg.norm(Xc - Xc.mean(axis=0), axis=1)
                manter[pos[np.argsort(score)[:n_alvo]]] = True

        est = self.base_estimator if self.base_estimator is not None else LogisticRegression(max_iter=2000)
        self.model_ = clone(est)
        self.model_.fit(X_scaled[manter], y[manter])
        return self

    def predict(self, X):
        X_scaled = self.scaler_.transform(X)
        return self.model_.predict(X_scaled)


def avaliar_embeddings(df, nome_tarefa="Tarefa", n_splits=5, random_state=42):
    feature_cols = [c for c in df.columns if c.startswith("f_")]
    X = df[feature_cols].values
    y = df["class_name"].values
    classes = np.unique(y)

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    estrategias = {
        "1. Class Weight (Padrão Recomendado)": None, # Usa todo o treino com pesos balanceados
        "2. Subamostragem Aleatória (no treino)": "aleatorio",
        "3. Menos Outlier - Isolation (no treino)": "isolation",
    }

    print(f"=== Avaliação Supervisionada: {nome_tarefa} (N={len(df)}) ===")

    for nome_est, metodo in estrategias.items():
        f1s, baccs = [], []
        recalls_por_classe = {c: [] for c in classes}

        for train_idx, val_idx in cv.split(X, y):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            if metodo is None:
                scaler = StandardScaler()
                X_tr_s = scaler.fit_transform(X_train)
                X_va_s = scaler.transform(X_val)
                clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state)
                clf.fit(X_tr_s, y_train)
                y_pred = clf.predict(X_va_s)
            else:
                clf = UnderSamplingClassifier(
                    base_estimator=LogisticRegression(max_iter=2000, random_state=random_state),
                    metodo=metodo,
                    random_state=random_state
                )
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_val)

            f1s.append(f1_score(y_val, y_pred, average="macro"))
            baccs.append(balanced_accuracy_score(y_val, y_pred))

            recs = recall_score(y_val, y_pred, labels=classes, average=None, zero_division=0)
            for c, r in zip(classes, recs):
                recalls_por_classe[c].append(r)

        print(f"\n[{nome_est}]")
        print(f"  Balanced Accuracy : {np.mean(baccs):.3f} ± {np.std(baccs):.3f}")
        print(f"  F1 Macro          : {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
        recall_str = ", ".join([f"{c}: {np.mean(recalls_por_classe[c]):.3f} (±{np.std(recalls_por_classe[c]):.3f})" for c in classes])
        print(f"  Recall por classe : {recall_str}")

# Exemplo de chamada para os seus dois DataFrames:
avaliar_embeddings(df_densidade, nome_tarefa="Densidade")
avaliar_embeddings(df_tamanho, nome_tarefa="Tamanho")

=== Avaliação Supervisionada: Densidade (N=1010) ===

[1. Class Weight (Padrão Recomendado)]
  Balanced Accuracy : 0.854 ± 0.033
  F1 Macro          : 0.851 ± 0.023
  Recall por classe : Denso: 0.780 (±0.072), Distribuido: 0.928 (±0.018)

[2. Subamostragem Aleatória (no treino)]
  Balanced Accuracy : 0.847 ± 0.029
  F1 Macro          : 0.798 ± 0.025
  Recall por classe : Denso: 0.866 (±0.061), Distribuido: 0.828 (±0.028)

[3. Menos Outlier - Isolation (no treino)]
  Balanced Accuracy : 0.755 ± 0.023
  F1 Macro          : 0.649 ± 0.025
  Recall por classe : Denso: 0.909 (±0.029), Distribuido: 0.600 (±0.036)
=== Avaliação Supervisionada: Tamanho (N=1010) ===

[1. Class Weight (Padrão Recomendado)]
  Balanced Accuracy : 0.728 ± 0.047
  F1 Macro          : 0.708 ± 0.037
  Recall por classe : Grande: 0.493 (±0.116), Medio: 0.931 (±0.020), Pequeno: 0.760 (±0.051)

[2. Subamostragem Aleatória (no treino)]
  Balanced Accuracy : 0.778 ± 0.031
  F1 Macro          : 0.498 ± 0.018
  Recall por cla

In [14]:
df_densidade_balan = balancear(df_densidade)
df_tamanho_balan = balancear(df_tamanho)

Antes:  {'Distribuido': 778, 'Denso': 232}
Depois (aleatorio): {'Denso': 232, 'Distribuido': 232}
Removidos: 546 de 1010

Antes:  {'Medio': 859, 'Pequeno': 125, 'Grande': 26}
Depois (aleatorio): {'Grande': 26, 'Medio': 26, 'Pequeno': 26}
Removidos: 932 de 1010



In [15]:
res_dens = analisar_dataset_yolo(df_densidade_balan, "Densidade", k_kmeans=2, k_hier=2)

Output hidden; open in https://colab.research.google.com to view.

In [16]:
res_tam  = analisar_dataset_yolo(df_tamanho_balan, "Tamanho", k_kmeans=3, k_hier=3)

Output hidden; open in https://colab.research.google.com to view.